# Étape 3 - Entraînement d'un modèle prédictif

On entraîne un premier modèle simple, on le **valide** avec une validation croisée
adaptée aux séries temporelles, puis on l'entraîne sur toutes les données.

## Modèle : forêt aléatoire

Une `RandomForestRegressor` = une moyenne de plusieurs arbres de décision.
Ici 10 arbres, avec une graine aléatoire fixée pour la reproductibilité.

## Validation croisée temporelle

On ne peut pas mélanger les dates au hasard (on prédirait le passé avec le futur).
`cross_validate` utilise `TimeSeriesSplit` : on entraîne sur un début de série,
on teste sur la suite, et on répète en agrandissant la fenêtre.

## 1. Importer les librairies

In [ ]:
import sys
sys.path.append('..')
import yaml
import logging
import logging.config
import numpy as np
import pandas as pd
pd.set_option('display.min_rows', 500)
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 500)
pd.set_option('max_colwidth', 400)

from foodcast.domain.transform import etl
from foodcast.domain.feature_engineering import features_offline, features_online
from foodcast.domain.forecast import span_future, cross_validate, plotly_predictions
from foodcast.domain.multi_model import MultiModel
from sklearn.ensemble import RandomForestRegressor
import foodcast.settings as settings
import plotly.graph_objects as go

with open(settings.LOGGING_CONFIGURATION_FILE, 'r') as f:
    logging.config.dictConfig(yaml.safe_load(f.read()))

%load_ext autoreload
%autoreload 2

## 2. Reprendre le jeu d'entraînement des étapes 1 et 2

In [ ]:
# Reprise des étapes 1 et 2
df = etl(settings.DATA_DIR, 197, 200)
df = features_offline(df)
x_train = df.drop(columns=['cash_in']).set_index('order_date')
y_train = df[['order_date', 'cash_in']].set_index('order_date')['cash_in']
x_train.head()

## 3. Créer le modèle

10 arbres, `random_state` fixé (42 ici, n'importe quelle valeur fixe convient).

In [ ]:
simple_model = RandomForestRegressor(n_estimators=10, random_state=42)
simple_model

## 4. Regarder le code de `cross_validate`

In [ ]:
cross_validate??

## 5. Valider le modèle (3 folds)

`cross_validate(model, x, y, n_fold)` renvoie :

- `maes` : un tableau des erreurs absolues moyennes (une par fold)
- `preds` : un dataframe des prédictions (colonne `y_pred_simple`)

In [ ]:
maes, preds = cross_validate(simple_model, x_train, y_train, n_fold=3)
maes

**Question — unité de la MAE ?** Des dollars (même unité que `cash_in`).

**Est-ce un bon indicateur ici ?** Pas vraiment : le chiffre d'affaires est nul la nuit
et très élevé le soir. Une MAE globale mélange ces régimes très différents ;
une erreur *relative* ou calculée par tranche horaire serait plus parlante.

## 6. Regarder le code de `plotly_predictions`

In [ ]:
plotly_predictions??

## 7. Tracer les prédictions de validation croisée face à la vérité

`plotly_predictions(preds, y)` : `preds` = prédictions, `y` = vraies valeurs.

In [ ]:
plotly_predictions(preds, y_train)

## 8. Entraîner le modèle sur **tout** le jeu d'entraînement

Une fois la performance jugée acceptable, on ré-entraîne sur 100 % des données
avec la méthode `fit` de scikit-learn.

In [ ]:
simple_model.fit(x_train, y_train)

Le modèle est prêt à prédire. Mais pour prédire le futur, il faut d'abord
construire le jeu de prédiction.

➡️ Étape suivante : `04_feature_engineering_online.ipynb`